# 1.4 Data Aggregation (tailored for demand prediction)

In this notebook we create a demand column in our dataset. This way we have our target variable which is essential for our data to be fully prepared for demand prediciton with ml models.

We have one big dataset with all trips now that we need to aggregate the data into different temporal resolutions, different spatial resolutions:

Temporal resolutions:
1. Hourly data
2. 2-hourly data
3. 6-hourly data
4. 24-hourly data

Spatial resolutions (h3):
1. high resolution (res = 8)
2. medium resolution (res = 5)
3. low resolution (res = 3)

Note that we want to have both the hexagon codes and the census tracts as data to compare performance differences in models between using the hexagon data and the census tract data.

In [1]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import numpy as np
from itertools import product

import h3
from shapely.geometry import shape
import geopandas as gpd
# import holidays

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load data                               #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

trips_all = pd.read_parquet("../data/CHICAGO_TAXI_TRIP_WEATHER.parquet")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Configurations                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

START_TIME = pd.to_datetime("2025-01-01 00:00:00")
END_TIME   = pd.to_datetime("2026-01-01 00:00:00")

BUCKET_COL = "hour_stamp_since_epoch"

TEMPORAL_RESOLUTIONS = ["1h", "2h", "6h", "24h"]

SPATIAL_RESOLUTIONS = {
    "low":    "pickup_h3_low_resolution",
    "medium": "pickup_h3_medium_resolution",
    "high":   "pickup_h3_high_resolution",
}

RESOLUTION_TO_H3 = {
    "low":    6,
    "medium": 7,
    "high":   8,
}

WEATHER_FEATURES = [
    "temperature_2m", "apparent_temperature", "precipitation",
    "snowfall", "snow_depth", "wind_speed_10m", "cloud_cover",
    "is_day", "rain", "sunshine_duration"
]

LEAKY_COLS = [
    "Trip ID", "Taxi ID",
    "Trip Start Timestamp", "Trip End Timestamp",
    "Trip Seconds", "Trip Miles",
    "Fare", "Tips", "Tolls", "Extras", "Trip Total",
    "Payment Type", "Company",
    "Pickup Census Tract", "Dropoff Census Tract",
    "Pickup Community Area", "Dropoff Community Area",
    "Pickup Centroid Latitude", "Pickup Centroid Longitude", "Pickup Centroid Location",
    "Dropoff Centroid Latitude", "Dropoff Centroid Longitude", "Dropoff Centroid  Location",
    "dropoff_h3_low_resolution", "dropoff_h3_medium_resolution", "dropoff_h3_high_resolution",
    "start_year", "date",
    "relative_humidity_2m", "surface_pressure", "wind_speed_100m", "direct_radiation",
]


In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Aggregate data                          #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

# All numeric columns to aggregate by mean
numeric_cols = trips_all.select_dtypes(include="number").columns.tolist()

for freq in TEMPORAL_RESOLUTIONS:
    trips_all["time_bucket"] = trips_all["Trip Start Timestamp"].dt.floor(freq)

    for spat_res, hex in SPATIAL_RESOLUTIONS.items():

        # trip_count = number of trips per time bucket AND hexagon
        trip_count = (
            trips_all
            .groupby(["time_bucket", hex])
            .size()
            .rename("trip_count")
        )

        # Mean of all numeric columns per time bucket and hexagon
        agg = (
            trips_all
            .groupby(["time_bucket", hex])[numeric_cols]
            .mean()
        )

        # Combine and reset index
        agg = agg.join(trip_count).reset_index()

        out = f"../data/aggregated/temporal_spatial_res/chicago_taxi_{freq}_{f"{spat_res}_res"}.parquet"
        agg.to_parquet(out, index=False)
        print(f"[{freq} | {f"{spat_res}"}]  shape={agg.shape}  saved → {out}")

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fill non-demand hours                   #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def fill_missing_hours(df, time_col="time_bucket", hex_col="h3_hexagon_low", start_time=START_TIME, end_time=END_TIME):

    df["hour_stamp_since_epoch"] = (
        df["time_bucket"]
        .dt.floor("1h")
        .astype("int64") // 10**6 // 3600  # datetime64[us] → seconds → hours
    )
    
    expected_hours = pd.date_range(start=start_time,
                                   end=end_time,
                                   freq="h")
    
    all_hexagons = df[hex_col].unique()
    
    # full cartesian product of all hexagons x all hours
    full_index = pd.MultiIndex.from_product(
        [all_hexagons, expected_hours],
        names=[hex_col, time_col]
    )
    
    df_full = full_index.to_frame(index=False)
    
    df = df_full.merge(df, on=[hex_col, time_col], how="left")
    
    n_missing = len(df_full) - len(df.dropna(subset=[c for c in df.columns if c not in [hex_col, time_col]]))
    print(f"Total expected rows : {len(df_full)}")
    print(f"Missing combinations: {n_missing}")
    
    df = df.sort_values([hex_col, time_col]).reset_index(drop=True)
    
    print(f"Done. New shape: {df.shape}")
    return df

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Non-demand hexagons                     #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

CITY_POLYGONS = {
    "chicago": {
        "type": "Polygon",
        "coordinates": [[
            [-87.94011, 41.64454],
            [-87.52414, 41.64454],
            [-87.52414, 42.02304],
            [-87.94011, 42.02304],
            [-87.94011, 41.64454],
        ]]
    },
}


def get_city_hexagons(city, resolution):
    """
    Get all H3 hexagons covering a city at a given resolution.

    Args:
        city:       City name as a lowercase string (e.g. "chicago").
        resolution: H3 resolution integer (e.g. 7, 8, 9).
    """
    # if city.lower() not in CITY_POLYGONS:
    #     raise ValueError(f"City '{city}' not found. Available cities: {list(CITY_POLYGONS.keys())}")

    polygon = CITY_POLYGONS[city.lower()]
    hexagons = list(h3.geo_to_cells(polygon, resolution))
    print(f"  Found {len(hexagons)} hexagons for {city} at resolution {resolution}")
    return hexagons


def add_non_demand_hex(df, hex_col, bucket_col, city, resolution):
    """
    Ensure every (time_bucket, hexagon) combination exists in the DataFrame,
    filling missing rows with trip_count=0 for hexagons with no demand.

    Args:
        df:         Pre-aggregated DataFrame with trip_count and hex/bucket columns.
        hex_col:    Column name for H3 hex cell identifier.
        bucket_col: Column name for time bucket.
        city:       City name as a lowercase string (e.g. "chicago").
        resolution: H3 resolution integer (e.g. 7, 8, 9).
    """

    city_hexagons = get_city_hexagons(city, resolution)
    all_buckets   = sorted(df[bucket_col].unique())

    full_grid = pd.DataFrame(
        list(product(all_buckets, city_hexagons)),
        columns=[bucket_col, hex_col]
    )

    df = full_grid.merge(df, on=[bucket_col, hex_col], how="left")
    df["trip_count"] = df["trip_count"].fillna(0).astype(int)

    # Forward fill numeric context columns (e.g. weather) per hexagon
    numeric_context = df.select_dtypes(include="number").columns.difference(["trip_count"])
    df[numeric_context] = df.groupby(hex_col)[numeric_context].ffill()

    # print(f"  Grid shape after fill: {df.shape}")
    return df

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# NaN Filling (weather, time_bucket)      #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

chicago_holidays = holidays.country_holidays("US", subdiv="IL")

def fill_time_bucket(df):
    """
    Exctract missing temporal data from hour_stam_since_epoch.
    """
    mask = df["time_bucket"].isna()
    df.loc[mask, "time_bucket"] = pd.to_datetime(
        df.loc[mask, "hour_stamp_since_epoch"] * 3600, unit="s"
    )

    df["start_month"] = df["time_bucket"].dt.month
    df["start_hour"]  = df["time_bucket"].dt.hour
    df["day_of_week"] = df["time_bucket"].dt.dayofweek
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)
    df["is_rush_hour"] = df["start_hour"].isin([7, 8, 9, 17, 18, 19]).astype(int)
    df["is_holiday"]  = df["time_bucket"].dt.date.map(lambda d: int(d in chicago_holidays))


    print(f"  Filled {mask.sum()} missing time_bucket values")
    return df


def fill_weather(df, bucket_col, weather_cols):
    """
    Fill missing weather data based on time_bucket, assuming city-wide uniform weather.

    Args:
        df:          DataFrame with weather columns and time_bucket.
        bucket_col:  Column name for time bucket.
        weather_cols: List of weather column names to fill.
    """
    weather_lookup = (
        df.dropna(subset=weather_cols)
        .groupby(bucket_col)[weather_cols]
        .first()
    )

    df = df.drop(columns=weather_cols).merge(weather_lookup, on=bucket_col, how="left")

    still_missing = df[weather_cols].isna().sum()
    
    # print(weather_lookup)

    print(f"  Weather NaNs remaining after fill:\n{still_missing}")
    return df

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Sine/Cosine Transformation              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

def sine_cosine_encode(df, col, period):
    """
    :param df: DataFrame containing the column to encode.
    :param col: Name of the column to encode (e.g. "hour", "month").
    :param period: The period of the cycle (e.g. 24 for hours, 12 for months).
    """
    df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / period)
    df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / period)
    return df

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Create datasets                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

datasets = {}

for temp_res in TEMPORAL_RESOLUTIONS:
    for spat_res, hex_col in SPATIAL_RESOLUTIONS.items():

        resolution = RESOLUTION_TO_H3[spat_res]

        agg_df = pd.read_parquet(f"../data/aggregated/temporal_spatial_res/chicago_taxi_{temp_res}_{spat_res}_res.parquet")
        print(f"\nBuilding [{temp_res} | {spat_res}] resolution using {hex_col}...")

        agg_df = fill_missing_hours(agg_df, time_col="time_bucket", hex_col=f"pickup_h3_{spat_res}_resolution", start_time=START_TIME, end_time=END_TIME)
        agg_df = add_non_demand_hex(agg_df, hex_col, BUCKET_COL, city="chicago", resolution=resolution)
        agg_df = fill_time_bucket(agg_df)
        agg_df = fill_weather(agg_df, bucket_col=BUCKET_COL, weather_cols=WEATHER_FEATURES)
        agg_df = sine_cosine_encode(agg_df, "start_hour", 24)
        agg_df = sine_cosine_encode(agg_df, "start_month", 12)
        agg_df = sine_cosine_encode(agg_df, "day_of_week", 7)

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Save datasets                           #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

for key, df in datasets.items():
    out = f"../data/aggregated/aggregated_demand_{key}.parquet"
    df.to_parquet(out, index=False)
    print(f"Saved → {out}")

In [ ]:
test = pd.read_parquet("../data/aggregated/aggregated_demand_2h_high.parquet")
print(len(test))
print(len(test[["h3_hexagon_high", "time_bucket"]].drop_duplicates()))
print("# of unique hexagons:", test["h3_hexagon_high"].nunique())
print("# of unique dates:", test["time_bucket"].dt.date.nunique())

In [ ]:
# check for missing hours in time_bucket
expected = pd.date_range(start="2025-01-01 00:00:00", 
                         end="2026-01-01 00:00:00", 
                         freq="2h")

missing = expected.difference(test["time_bucket"])

print(f"Missing hours: {len(missing)}")